# Ontology Repair Evaluation

This notebook evaluates and compares ontology repair plan selection strategies. It covers:
1. LLM-based repair ranking – uses a large language model to rank and select repair plans suggested by LogMap.
2.	LogMap heuristic selection – applies LogMap’s built-in strategy: remove plans with the highest conflict score, using lowest confidence as a tie-break.
3.	Evaluation – measures performance of selected repair plans against reference alignments using standard metrics (TP, TN, FP, FN, etc.).

**Why Repair is Used:**

Ontology repair resolves conflicting or incorrect mappings between ontologies to improve alignment quality. It ensures that integrated knowledge bases are consistent, reliable, and more accurate for downstream applications like data integration, reasoning, or biomedical research.

**Main Goal:**

Reduce false negatives (FN) and increase true negatives (TN) for higher-quality ontology alignment.

\
**Note:** The pipeline automatically stores all intermediate and generated results in the `output/` directory, while finalized and manually curated results are saved in the `final_results/` directory.

### LLM-based Repair Ranking
Runs LLM semantic ranking of LogMap repair plans for each dataset subset and saves the selected repair decisions for evaluation.

In [ ]:
from pathlib import Path
from utils.utils import format_subsets_ontologies_paths
from modules.logical_repairer import LogicalRepairer
from utils.llm_server.open_router import OpenRouterServer

if __name__ == "__main__":

    ALL_DATASET_NAMES = {
        "anatomy": ["human-mouse"],
        "bioml-2024": [
            "snomed-fma.body", 
            "snomed-ncit.neoplas", 
            "snomed-ncit.pharm",
            "ncit-doid", 
        ],
        "largebio": [
            "fma-snomed", 
            "snomed-nci", 
            "fma-nci"
            ],
    }

    REPAIR_PLAN_FILE_MAP = {
        "human-mouse": "anatomy_repairs.txt",
        "ncit-doid": "ncit-doid_repairs.txt",
        "snomed-ncit.pharm": "snomed-ncit-pharma_repairs.txt",
        "snomed-ncit.neoplas": "snomed-ncit-neoplas_repairs.txt",
        "snomed-fma.body": "snomed-fma-organ_repairs.txt",
        "fma-snomed": "largebio-fma-snomed_repairs.txt",
        "fma-nci": "largebio-fma-nci_repairs.txt",
        "snomed-nci": "largebio-snomed-nci_repairs.txt",
    }

    repair_plans_path = Path("data/LogMap_repair_plans")
    llm_server = OpenRouterServer()
    model = "qwen/qwen3-vl-8b-instruct"

    for dataset_name, subsets in ALL_DATASET_NAMES.items():
        for subset in subsets:

            print(f"[INFO] Processing dataset '{dataset_name}', subset '{subset}'")

            o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subset)
            repair_file = REPAIR_PLAN_FILE_MAP.get(subset)

            if not repair_file:
                print(f"[WARNING] No repair plan found for '{subset}', skipping...")
                continue

            repairer = LogicalRepairer(
                o1_path=o1_path,
                o2_path=o2_path,
                use_prebuilt_prompts=False,
                llm_server=llm_server,
                model=model
            )

            repair_path = repair_plans_path / repair_file

            print(f"[INFO] Running LLM-based repair ranking for '{subset}' using {repair_path}")

            repairer.run_reduced(
                logmap_output_path=str(repair_path),
                dataset_name=f"{dataset_name}/{subset}",
                experiment_type="llm_semantic_ranking",
                save=True,
            )

            print(f"[INFO] Finished LLM ranking for '{subset}'\n")

### Evaluate LLM Repair Decisions

Loads LLM-selected repair plans, converts them to a structured format, and evaluates their alignment against ground truth for each dataset subset.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import os
from modules.evaluator import OntologyAlignmentEvaluator

if __name__ == "__main__":

    ALL_DATASET_NAMES = {
        "anatomy": ["human-mouse"],
        "bioml-2024": [
            "snomed-fma.body", 
            "snomed-ncit.neoplas", 
            "snomed-ncit.pharm",
            "ncit-doid", 
        ],
        "largebio": [
            "fma-snomed", 
            "snomed-nci", 
            "fma-nci"
            ],
    }

    model = "qwen3-vl-8b-instruct"

    final_results_root = Path("final_results")

    for dataset_name, subsets in ALL_DATASET_NAMES.items():
        for subset in subsets:
        
            print(f"[INFO] Processing dataset '{dataset_name}', subset '{subset}'")

            final_results = final_results_root / "repair_ontologies" / "repair_reduced_info_llm_output"/ model / dataset_name / subset / "repair_results.json"
            gt_path = f"data/{dataset_name}/{subset}/refs_equiv/full.tsv"

            with open(final_results, "r", encoding="utf-8") as f:
                results_data = json.load(f)

            results_list = []

            for conflict in results_data:
                plan_id = conflict["selected_plan"]
                selected_plan = next(p for p in conflict["plans"] if p["plan_id"] == plan_id)
                
                for mapping in selected_plan["mappings"]:
                    results_list.append({
                        "Source": mapping["source_uri"],
                        "Target": mapping["target_uri"],
                        "SelectedPlan": plan_id,
                        "DecisionType": conflict.get("decision_type"),
                        "Reasoning": conflict.get("reasoning")
                    })

            # SAVE RESULTS
            df = pd.DataFrame(results_list)
            df["LLMDecision"] = False

            output_dir = f"output/repair_evaluation_results/{model}/{dataset_name}/{subset}"
            os.makedirs(output_dir, exist_ok=True)

            # EVALUATION
            evaluator = OntologyAlignmentEvaluator(gt_path)

            report = evaluator.evaluate(
                df=df,
                dataset_name=dataset_name,
                experiment_type=f"repair_reduced",
                prompts_used="repair",
                results_dir=output_dir
            )

            print("\n=== Evaluation Report ===")
            print(report)

[INFO] Processing dataset 'anatomy', subset 'human-mouse'

=== Evaluation Report ===
{'LLM': {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'TP': 0, 'TN': 2, 'FP': 0, 'FN': 1, 'Sensitivity': 0.0, 'Specificity': 1.0, 'YoudenIndex': 0.0, 'ConfusionMatrix': array([[2, 0],
       [1, 0]])}}
[INFO] Processing dataset 'bioml-2024', subset 'snomed-fma.body'

=== Evaluation Report ===
{'LLM': {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'TP': 0, 'TN': 432, 'FP': 0, 'FN': 1351, 'Sensitivity': 0.0, 'Specificity': 1.0, 'YoudenIndex': 0.0, 'ConfusionMatrix': array([[ 432,    0],
       [1351,    0]])}}
[INFO] Processing dataset 'bioml-2024', subset 'snomed-ncit.neoplas'

=== Evaluation Report ===
{'LLM': {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'TP': 0, 'TN': 352, 'FP': 0, 'FN': 16, 'Sensitivity': 0.0, 'Specificity': 1.0, 'YoudenIndex': 0.0, 'ConfusionMatrix': array([[352,   0],
       [ 16,   0]])}}
[INFO] Processing dataset 'bioml-2024', subset 'snomed-ncit.pharm'

=== Evaluation Report ===
{'

### Evaluate LogMap Heuristic

Applies LogMap’s plan-selection rules (highest conflict score, lowest confidence tie-break) to repair outputs, structures the results, and evaluates them against ground truth alignments for each dataset subset.

In [ ]:
from pathlib import Path
import pandas as pd
import os

from modules.evaluator import OntologyAlignmentEvaluator
from utils.prompts.logical_repair_prompt import (
    parse_logmap_repair_output_light
)

# LogMap selection heuristic
def select_logmap_plan_to_remove(plans):
    """
    LogMap chooses to remove the plan with:
        1. highest conflict score
        2. lowest confidence (If two plans have the same conflict score, LogMap removes the one it trusts LESS.)
    """
    return sorted(
        plans,
        key=lambda p: (-p["conflict_score"], p["confidence"])
    )[0]



if __name__ == "__main__":

    ALL_DATASET_NAMES = {
        "anatomy": ["human-mouse"],
        "bioml-2024": [
            "snomed-fma.body", 
            "snomed-ncit.neoplas", 
            "snomed-ncit.pharm",
            "ncit-doid", 
        ],
        "largebio": [
            "fma-snomed", 
            "snomed-nci", 
            "fma-nci"
            ],
    }

    REPAIR_PLAN_FILE_MAP = {
        "human-mouse": "anatomy_repairs.txt",
        "ncit-doid": "ncit-doid_repairs.txt",
        "snomed-ncit.pharm": "snomed-ncit-pharma_repairs.txt",
        "snomed-ncit.neoplas": "snomed-ncit-neoplas_repairs.txt",
        "snomed-fma.body": "snomed-fma-organ_repairs.txt",
        "fma-snomed": "largebio-fma-snomed_repairs.txt",
        "fma-nci": "largebio-fma-nci_repairs.txt",
        "snomed-nci": "largebio-snomed-nci_repairs.txt",
    }

    repair_plans_path = Path("data/LogMap_repair_plans")

    # LOOP DATASETS
    for dataset_name, subsets in ALL_DATASET_NAMES.items():
        for subset in subsets:

            print(f"[INFO] Processing {dataset_name} / {subset}")

            repair_file = REPAIR_PLAN_FILE_MAP.get(subset)
            if not repair_file:
                print(f"[WARNING] No repair plan for {subset}")
                continue

            repair_path = repair_plans_path / repair_file

            conflicts = parse_logmap_repair_output_light(
                str(repair_path)
            )

            results_list = []

            # Apply LogMap heuristic
            for conflict in conflicts:

                if not conflict["plans"]:
                    continue

                best_plan = select_logmap_plan_to_remove(conflict["plans"])

                for mapping in best_plan["mappings"]:
                    results_list.append({
                        "Source": mapping["source_uri"],
                        "Target": mapping["target_uri"],
                        "ConflictScore": best_plan["conflict_score"],
                        "Confidence": best_plan["confidence"],
                        "LogMapDecision": False
                    })

            df = pd.DataFrame(results_list)

            # SAVE predictions
            output_dir = f"output/logmap_repair_plan_evaluation/{dataset_name}/{subset}"
            os.makedirs(output_dir, exist_ok=True)

            gt_path = f"data/{dataset_name}/{subset}/refs_equiv/full.tsv"

            evaluator = OntologyAlignmentEvaluator(gt_path)

            report = evaluator.evaluate(
                df=df,
                dataset_name=dataset_name,
                experiment_type="logmap_plan_selection",
                prompts_used="logmap_txt",
                second_system_name="LogMapPlan",
                second_system_pred_col="LogMapDecision",
                results_dir=output_dir
            )

            print("\n=== Evaluation Report ===")
            print(report)

[INFO] Processing anatomy / human-mouse

=== Evaluation Report ===
{'LogMapPlan': {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'TP': 0, 'TN': 2, 'FP': 0, 'FN': 1, 'Sensitivity': 0.0, 'Specificity': 1.0, 'YoudenIndex': 0.0, 'ConfusionMatrix': array([[2, 0],
       [1, 0]])}}
[INFO] Processing bioml-2024 / snomed-fma.body

=== Evaluation Report ===
{'LogMapPlan': {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'TP': 0, 'TN': 447, 'FP': 0, 'FN': 1336, 'Sensitivity': 0.0, 'Specificity': 1.0, 'YoudenIndex': 0.0, 'ConfusionMatrix': array([[ 447,    0],
       [1336,    0]])}}
[INFO] Processing bioml-2024 / snomed-ncit.neoplas

=== Evaluation Report ===
{'LogMapPlan': {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'TP': 0, 'TN': 358, 'FP': 0, 'FN': 10, 'Sensitivity': 0.0, 'Specificity': 1.0, 'YoudenIndex': 0.0, 'ConfusionMatrix': array([[358,   0],
       [ 10,   0]])}}
[INFO] Processing bioml-2024 / snomed-ncit.pharm

=== Evaluation Report ===
{'LogMapPlan': {'Precision': 0.0, 'Recall': 0.0, 'F1'